# Test de Conectividad
Verifica la conexión a **MySQL** y **MinIO** desde Jupyter.

In [ ]:
import os

# Variables de entorno inyectadas por docker-compose
MYSQL_HOST     = os.getenv('MYSQL_HOST', 'mysql')
MYSQL_PORT     = int(os.getenv('MYSQL_PORT', 3306))
MYSQL_USER     = os.getenv('MYSQL_USER', 'user')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', 'user1234')
MYSQL_DATABASE = os.getenv('MYSQL_DATABASE', 'mydatabase')

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'minio:9000')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minio_user')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minio_password')

print('Variables cargadas correctamente')

## 1. Conexión a MySQL

In [ ]:
import mysql.connector

try:
    conn = mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DATABASE,
    )
    cursor = conn.cursor()

    cursor.execute('SELECT VERSION()')
    version = cursor.fetchone()
    print(f'Conectado a MySQL {version[0]}')

    cursor.execute('SHOW TABLES')
    tables = [row[0] for row in cursor.fetchall()]
    print(f'Tablas en "{MYSQL_DATABASE}": {tables if tables else "(vacía)"}')

    cursor.close()
    conn.close()
    print('Conexión cerrada correctamente')
except Exception as e:
    print(f'ERROR MySQL: {e}')

## 2. Conexión a MinIO (via boto3 / S3)

In [ ]:
import boto3
from botocore.client import Config

try:
    s3 = boto3.client(
        's3',
        endpoint_url=f'http://{MINIO_ENDPOINT}',
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY,
        config=Config(signature_version='s3v4'),
        region_name='us-east-1',
    )

    buckets = s3.list_buckets().get('Buckets', [])
    print(f'Conectado a MinIO en {MINIO_ENDPOINT}')
    print(f'Buckets: {[b["Name"] for b in buckets] if buckets else "(ninguno)"}')
except Exception as e:
    print(f'ERROR MinIO: {e}')

## 3. Prueba de escritura y lectura en MinIO

In [ ]:
import io

BUCKET = 'test-bucket'
KEY    = 'test/hello.txt'

try:
    # Crear bucket si no existe
    existing = [b['Name'] for b in s3.list_buckets().get('Buckets', [])]
    if BUCKET not in existing:
        s3.create_bucket(Bucket=BUCKET)
        print(f'Bucket "{BUCKET}" creado')
    else:
        print(f'Bucket "{BUCKET}" ya existe')

    # Escribir objeto
    s3.put_object(Bucket=BUCKET, Key=KEY, Body=b'Hola desde Jupyter!')
    print(f'Objeto "{KEY}" escrito')

    # Leer objeto
    response = s3.get_object(Bucket=BUCKET, Key=KEY)
    content = response['Body'].read().decode()
    print(f'Objeto leído: "{content}"')
except Exception as e:
    print(f'ERROR escritura/lectura MinIO: {e}')